In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    print("API Key is not found")

MODEL = "gpt-4o-mini"

In [10]:
SUMMARIZER_AGENT_PROMPT = f"""
You are a research assistant. You work inside the {SANDBOX_DIR} directory.
You have two MCP servers filesystem(list/read/search/write files etc) and fetch(get web url data)
If user ask for some information, use fetch to get that page. Summarize it and save as an md file.
"""

In [16]:
SUMMARIZER_AGENT_PROMPT = f"""
You are a research assistant. You work inside the {SANDBOX_DIR} directory.
You have two MCP servers filesystem(list/read/search/write files etc) and fetch(get web url data)

Always start by listing the directory and reading every local file that could be relavant - 
dont assume the user named all the files that matter.

If user ask for some information, use fetch to get that page. Summarize it and save as an md file.

Actually do the work at every step - never just describe what you plan to do and stop.
Recheck the user instructions - you are only done when the requirments are fully satisfied.
your final message should be you reporting back to confirm that instructions have been fully Fullfilled.

CRITICAL: If the user asks you to write a file, DO NOT ask the user's permission to write the file.
Proceed as instructed, saving the file is MANDATORY.
"""

In [17]:
fetch_server_params = {
    "command": "uvx",
    "args" : ["mcp-server-fetch"]
}

In [18]:
filesystem_server_params = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", SANDBOX_DIR]
}

In [20]:
async with MCPServerStdio(name="Filesystem Server", params=filesystem_server_params,
                          client_session_timeout_seconds=60) as filesystem_server:
    fs_tools = await filesystem_server.list_tools()
    
    print(f"✅ MCP server Connected. The server offers {len(fs_tools)} tool(s):\n")
    for tool in fs_tools:
        print(f"🛠️ {tool.name} --   {tool.description.strip().splitlines()[0]}")
    async with MCPServerStdio(name="Fetch Server", params=fetch_server_params,
                              client_session_timeout_seconds=60) as fetch_server:
        tools = await fetch_server.list_tools()
        print(f"\n✅ Connected. the server offer {len(tools)} tools: \n")
        for tool in tools:
            print(f"🛠️ {tool.name} --   {tool.description.strip().splitlines()[0]}")
        
        summarizer_agent = Agent(
            name = "Summarizer Agent",
            instructions= SUMMARIZER_AGENT_PROMPT,
            model = MODEL,
            mcp_servers=[filesystem_server, fetch_server]
        )

        with trace("Summarizer Agent Run"):
            result = await Runner.run(
                summarizer_agent,
                input = "Read the project Falcon notes to understand what we are building. The fetch ."\
                    "https://news.ycombinator.com/ - and complete the outstanding todo item" \
                    "Write your findings in a competetive teardown in the project folder",
                max_turns=10
            )


    print(f"Last Agent: {result.last_agent.name}")
    print("------")
    print(result.final_output)

✅ MCP server Connected. The server offers 14 tool(s):

🛠️ read_file --   Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.
🛠️ read_text_file --   Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within allowed directories.
🛠️ read_media_file --   Read a file and return it as a base64-encoded content block with its MIME type. Image and audio files are returned as image/audio content; any other file type is returned as an embedded resource. Only works within allowed directories.
🛠️ read_multiple_files --   Read the contents of multiple files simultaneously. This is more efficien